In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

from functools import partial

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)

In [2]:
@jax.jit
def test_Sjax(params, fields):
    phi = fields['phi']
    S = phi**2
    
    for ax in range(phi.d):
        S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

    S += params['lambda'] * (phi**2 - 1)**2
    return S

@jax.jit
def test_Sjax_2(kappa, lamb, phi):
    S = phi**2
    
    for ax in range(phi.d):
        S -= 2 * kappa * phi * phi.nn_field(ax)

    S += lamb * (phi**2 - 1)**2
    return S    

def test_Sjax_3(params, fields):
    return test_Sjax_2(params['kappa'], params['lambda'], fields['phi'])

@jax.jit
def test_Sjax_4(params, fields):
    phi = fields[0]
    S = phi**2
    
    for ax in range(phi.d):
        S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

    S += params['lambda'] * (phi**2 - 1)**2
    return S

L4 = lat.SquareLattice(dims=(4,4))
L6 = lat.SquareLattice(dims=(6,6))
phi4 = lat.LatticeField(L4).unit_fill()

Stest = test_Sjax(params={'kappa': 0.11, 'lambda': 1.0}, fields={'phi': phi4})
print(Stest.F)
print(jnp.sum(Stest.F))

Stest2 = test_Sjax_2(0.11, 1.0, phi4)
print(Stest2.F)
print(jnp.sum(Stest2.F))

[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
8.96
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
8.96


I0000 00:00:1703281383.799477       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [3]:
%timeit test_Sjax(params={'kappa': 0.11, 'lambda': 1.0}, fields={'phi': phi4})


8.57 µs ± 16.1 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [4]:
%timeit test_Sjax_2(0.11, 1.0, phi4)


7.94 µs ± 23.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [5]:
%timeit test_Sjax_3(params={'kappa': 0.11, 'lambda': 1.0}, fields={'phi': phi4})


7.43 µs ± 28.1 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [6]:
%timeit test_Sjax_4(params={'kappa': 0.11, 'lambda': 1.0}, fields=[phi4])


7.81 µs ± 64.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [47]:
U = set([1,2,3])
V = set([2,4,6])
print(U ^ V, U & V, U | V)
U |= V
print(U)

{1, 3, 4, 6} {2} {1, 2, 3, 4, 6}
{1, 2, 3, 4, 6}


In [63]:
test_Sjax_2(0.11, 0.4, phi4).F

test_grad = jax.jit(jax.grad(lambda kappa, lamb, phi: jnp.sum(test_Sjax_2(kappa, lamb, phi).F), argnums=2))
test_grad(0.11, 0.4, phi4).F

Array([[1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12]], dtype=float64)

In [68]:
%timeit test_grad(0.11, 0.4, phi4)

7.5 µs ± 13.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [72]:
@jax.jit
def test_Sjax_2_tot(kappa, lamb, phi):
    return jnp.sum(test_Sjax_2(kappa, lamb, phi).F)

test_grad_2 = jax.jit(jax.grad(test_Sjax_2_tot, argnums=2))
test_grad_2(0.11, 0.4, phi4).F

Array([[1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12]], dtype=float64)

In [70]:
%timeit test_grad_2(0.11, 0.4, phi4)

7.46 µs ± 64.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [84]:
def test_Sjax_3_tot(params, fields):
    return jnp.sum(test_Sjax(params, fields).F)

test_grad_3 = jax.jit(jax.grad(test_Sjax_3_tot, argnums=1))
test_grad_3({'kappa': 0.11, 'lambda': 1.0}, {'phi': phi4})['phi'].F

Array([[1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12],
       [1.12, 1.12, 1.12, 1.12]], dtype=float64)

In [85]:
%timeit test_grad_3({'kappa': 0.11, 'lambda': 1.0}, {'phi': phi4})

8.13 µs ± 17.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [5]:
# Define the action
class ScalarAction(lhmc.Action):        

    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S
    
    # Exact force function instead of autodiff, for testing purposes
    @staticmethod
    @jax.jit
    def exact_force(fields, params):
        phi = fields[0]

        J = phi.nn_field(axis=0, shift=1) + phi.nn_field(axis=0, shift=-1)
        for ax in range(1,phi.d):
            J += phi.nn_field(axis=ax, shift=1)
            J += phi.nn_field(axis=ax, shift=-1)

        F = -2 * params['kappa'] * J
        F += 2 * phi.F
        F += 4 * params['lambda'] * (phi.F**2 - 1) * phi.F

        return F

# Component actions for testing composition
class ScalarKineticAction(lhmc.Action):
    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2

        for ax in range(phi.d):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        return S

class ScalarQuarticInt(lhmc.Action):    
    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        return params['lambda'] * (phi**2 - 1)**2




In [16]:
# Actions depend on parameters, fields are arbitrary including dimension
S_A = ScalarAction(params = {'kappa': 0.18, 'lambda': 1.0}, field_names=['phi'])
S_B = ScalarAction(params = {'kappa': 0.17, 'lambda': 1.0}, field_names=['phi'])

# Force calculation should work automatically, without explicit definition above
F_A = S_A.get_forces()
print(F_A)


# Calling the action object as a function, using fields as input, should call and return the summed action functional
L4 = lat.SquareLattice(dims=(4,4))
L6 = lat.SquareLattice(dims=(6,6))
phi4 = lat.LatticeField(L4).unit_fill()
phi6 = lat.LatticeField(L6).unit_fill()
SA_total_4 = S_A.S({'phi': phi4})
SA_total_6 = S_A.S({'phi': phi6})
SB_total_4 = S_B.S({'phi': phi4})

print(SA_total_4, SA_total_6, SB_total_4)

<PjitFunction of <function Action._compute_forces.<locals>.<lambda> at 0x1391082c0>>
4.48 10.08 5.119999999999998


In [17]:
S_A.S({'phi': phi4, 'phi2': phi4})

Array(4.48, dtype=float64)

In [18]:
print(S_A.field_names)
print(S_A.sub_actions)

['phi']
[]


In [9]:
%timeit S_A.S({'phi': phi4})

%timeit jnp.sum(test_Sjax(params={'kappa': 0.18, 'lambda': 1.0}, fields={'phi': phi4}).F)


14.7 µs ± 64.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
14.5 µs ± 59.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [10]:
print(F_A({'phi': phi4})['phi'].F)
print(S_A.exact_force(fields=[phi4], params=S_A.params).F)

%timeit F_A({'phi': phi4})
%timeit S_A.exact_force(fields=[phi4], params=S_A.params)

[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
5.91 µs ± 26.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
8.43 µs ± 43.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [12]:
# We should be able to build a joint action with *shared* fields, by composition.
S_A_kin = ScalarKineticAction(params={'kappa': 0.18}, field_names=['chi'])
S_A_int = ScalarQuarticInt(params={'lambda': 1.0}, field_names=['chi'])

S_A_joint = S_A_kin + S_A_int
print(S_A_joint.S({'chi': phi4}))  # Should match cell above
      
# On the other hand, we should also be able to compose multiple copies of a given action,
# without making the fields shared.
S_A_different = ScalarAction(params={'kappa': 0.18, 'lambda': 1.0}, field_names=['phi2'])
S_A_disjoint = S_A + S_A_different

print(S_A_disjoint.S(fields={'phi':phi4, 'phi2': phi4}))  # Should be 2x the action above

4.48
8.96


In [15]:
# Some edge cases/error modes:

# Trying to combine actions with two different field dimensions should error
try:
    print(S_A_disjoint.S(fields={'phi': phi4, 'phi2': phi6}))
except Exception as e:
    print(e)


add got incompatible shapes for broadcasting: (4, 4), (6, 6).
